<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [ ]:
class Search(BaseModel):
    """
    Search class to warp the search and results
    """
    search_tag:str  = 'Title/Abstract' #Tag to specifiy the search, can be any from pubmed, Defaul: Title/Abstract
    retmax:int = 200 #Maximum number of results to be retrieved
    retmode:str ='xml' #Format of the returned data, options are xml, 
    sort:str='relevance' #Way to sort the results
    mindate: int | None = None #Initial data to be search from, year
    maxdate: int | None = None #Final data to be search from, year
    idlist: List[int] | None = None
    email:str | None = None
    api_key:str | None = None
    country: str | None = None
    
    @model_validator(mode='before')
    def validate_email(cls,values:dict )->dict:
        email = get_from_dict_or_env(
            values, "email", "EMAIL"
        )
        values["email"] = email
        
        api_key = get_from_dict_or_env(values, 'api_key', 'API_KEY')
        values['api_key'] = api_key
        return values
        
    @field_validator('search_tag', mode='before')
    @classmethod
    def validate_search_tag(cls, v):
        if not v:
            v = 'Title/Abstract'
        if v not in SEARCH_TAGS.keys():
            raise ValueError(f'Search tag need to be some of {SEARCH_TAGS.keys()}')
        return SEARCH_TAGS[v]

In [1]:
#| echo: false
#| output: asis
show_doc(Search)

---

[source](https://github.com/Dmaturana81/pubmed_lib/blob/main/pubmed_lib/search.py#L21){target="_blank" style="float:right; font-size:smaller"}

### Search

>      Search (search_tag:str='Title/Abstract', retmax:int=200,
>              retmode:str='xml', sort:str='relevance', mindate:int|None=None,
>              maxdate:int|None=None, idlist:Optional[List[int]]=None,
>              email:str|None=None, api_key:str|None=None,
>              country:str|None=None)

Search class to warp the search and results

In [ ]:
@patch()
def search(
    self:Search,
    query: str, #Query to be search in pubmed
):
    """
    It receive a query to be searched in pubmed and return the handler of the search
    """
    Entrez.email = self.email
    Entrez.api_key = self.api_key
    query = self._generate_query(query)
    handle = Entrez.esearch(db='pubmed',
                    sort=self.sort,
                    retmax=self.retmax,
                    retmode=self.retmode,
                    term=query,
                    mindate = self.mindate,
                    maxdate =self. maxdate)
    results = Entrez.read(handle)
    return results['IdList']

In [2]:
#| echo: false
#| output: asis
show_doc(Search.search)

---

[source](https://github.com/Dmaturana81/pubmed_lib/blob/main/pubmed_lib/search.py#L73){target="_blank" style="float:right; font-size:smaller"}

### Search.search

>      Search.search (query:str)

It receive a query to be searched in pubmed and return the handler of the search

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| query | str | Query to be search in pubmed |

In [ ]:
search = Search(country='Brasil', mindate=2020, maxdate=2024)

In [ ]:
search

Search(search_tag='Title/Abstract', retmax=200, retmode='xml', sort='relevance', mindate=2020, maxdate=2024, idlist=None, email='elmaturana@gmail.com', api_key='3cfd60b4f78696d27f1c4df78d3fe6f90a09', country='Brasil')

In [ ]:
idlist = search.search('Protein stability')
idlist

['34137435', '33905626', '32492183', '37316640']

In [ ]:
search._generate_query('Protein stability')

'"Protein stability"[tiab] AND "Brasil"[ad]'

In [3]:
#| echo: false
#| output: asis
show_doc(Search.fetch_details)

---

[source](https://github.com/Dmaturana81/pubmed_lib/blob/main/pubmed_lib/search.py#L95){target="_blank" style="float:right; font-size:smaller"}

### Search.fetch_details

>      Search.fetch_details (idlist:List[int])

It receive a list of pubmedIds from a search, and retrieve all the details of those publications

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| idlist | List | list of pubmedid to be retreived |

In [ ]:
@patch
def results(
    self:Search,
    query:str, #Term to be queried in pubmed
)->list:
    """
    Method that do the search and retrieve a generator with all the infomration of the articles"""
    results = Results()
    # id_list = self.search(query)
    if id_list := self.search(query):
        articles = self.fetch_details(id_list)
        for article in articles:
            article_dict = parse_paperinfo(article)
            results.append( Result.model_validate(article_dict))
    return results

In [4]:
#| echo: false
#| output: asis
show_doc(Search.results)

---

[source](https://github.com/Dmaturana81/pubmed_lib/blob/main/pubmed_lib/search.py#L111){target="_blank" style="float:right; font-size:smaller"}

### Search.results

>      Search.results (query:str)

Method that do the search and retrieve a generator with all the infomration of the articles

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| query | str | Term to be queried in pubmed |
| **Returns** | **list** |  |